In [ ]:
import pandas as pd
import os
import logging


# ============================================================
# CONFIG
# ============================================================

RANDOM_STATE = 42
N_S1 = 1000

# Number of extra/random S2/S3 records relative to true matches
NEGATIVE_MULTIPLIER = 5

TRAIN_DIR = "../student_resource/dataset/train/"
OUTPUT_DIR = "sampled_data"


# ============================================================
# LOGGING SETUP
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S"
)

log = logging.getLogger(__name__)


def section(title):
    log.info("")
    log.info("=" * 70)
    log.info(title)
    log.info("=" * 70)


# ============================================================
# 1. LOAD DATA
# ============================================================

section("LOADING DATA")

log.info("Reading Source 1...")
s1 = pd.read_csv(
    f"{TRAIN_DIR}/train_source1.tsv",
    sep="\t"
)

log.info("Reading Source 2...")
s2 = pd.read_csv(
    f"{TRAIN_DIR}/train_source2.tsv",
    sep="\t"
)

log.info("Reading Source 3...")
s3 = pd.read_csv(
    f"{TRAIN_DIR}/train_source3.tsv",
    sep="\t"
)

log.info("Reading ground truth...")
ground_truth = pd.read_csv(
    f"{TRAIN_DIR}/train_ground_truth.tsv",
    sep="\t"
)

log.info("Loaded:")
log.info(f"  S1 records:          {len(s1):,}")
log.info(f"  S2 records:          {len(s2):,}")
log.info(f"  S3 records:          {len(s3):,}")
log.info(f"  Ground-truth rows:   {len(ground_truth):,}")


# ============================================================
# 2. SAMPLE S1
# ============================================================

section("SAMPLING SOURCE 1")

sample_size = min(N_S1, len(s1))

log.info(
    f"Sampling {sample_size:,} S1 records "
    f"(random_state={RANDOM_STATE})..."
)

sample_s1 = s1.sample(
    n=sample_size,
    random_state=RANDOM_STATE
).copy()

sample_s1_ids = set(sample_s1["entity_id"])

log.info(f"Sampled S1 records: {len(sample_s1):,}")


# ============================================================
# 3. GET GROUND TRUTH FOR SAMPLED S1
# ============================================================

section("FILTERING GROUND TRUTH")

log.info(
    "Finding ground-truth rows corresponding to "
    "the sampled S1 records..."
)

sample_gt = ground_truth[
    ground_truth["source1_entity_id"].isin(sample_s1_ids)
].copy()

log.info(f"Ground-truth rows found: {len(sample_gt):,}")

missing_gt = sample_s1_ids - set(
    sample_gt["source1_entity_id"]
)

if missing_gt:
    log.warning(
        f"{len(missing_gt):,} sampled S1 records have "
        f"no ground-truth row!"
    )
else:
    log.info(
        "✓ Every sampled S1 record has a ground-truth row."
    )


# ============================================================
# 4. IDENTIFY SINGLETONS
# ============================================================

section("ANALYSING SINGLETONS")

matched_values = (
    sample_gt["matched_entity_ids"]
    .fillna("")
    .astype(str)
    .str.strip()
)

singleton_mask = matched_values.eq("")

n_singletons = singleton_mask.sum()
n_non_singletons = (~singleton_mask).sum()

log.info(f"Sampled S1 records:       {len(sample_gt):,}")
log.info(f"Singleton S1 records:     {n_singletons:,}")
log.info(f"Non-singleton S1 records: {n_non_singletons:,}")

log.info(
    f"Singleton percentage: "
    f"{n_singletons / len(sample_gt) * 100:.2f}%"
)


# ============================================================
# 5. EXTRACT TRUE S2/S3 IDS
# ============================================================

section("EXTRACTING TRUE MATCH IDS")

true_s2_ids = set()
true_s3_ids = set()

for value in sample_gt["matched_entity_ids"]:

    if pd.isna(value):
        continue

    value = str(value).strip()

    if value == "":
        continue

    ids = [
        x.strip()
        for x in value.split(",")
        if x.strip()
    ]

    for entity_id in ids:

        if entity_id.startswith("S2-"):
            true_s2_ids.add(entity_id)

        elif entity_id.startswith("S3-"):
            true_s3_ids.add(entity_id)

        else:
            log.warning(
                f"Unexpected entity ID in ground truth: "
                f"{entity_id}"
            )


log.info(
    f"Unique true S2 matches: {len(true_s2_ids):,}"
)

log.info(
    f"Unique true S3 matches: {len(true_s3_ids):,}"
)

log.info(
    f"Total unique true S2/S3 records: "
    f"{len(true_s2_ids) + len(true_s3_ids):,}"
)


# ============================================================
# 6. GET TRUE MATCH RECORDS
# ============================================================

section("COLLECTING TRUE MATCH RECORDS")

matched_s2 = s2[
    s2["entity_id"].isin(true_s2_ids)
].copy()

matched_s3 = s3[
    s3["entity_id"].isin(true_s3_ids)
].copy()

log.info(
    f"True S2 records retrieved: "
    f"{len(matched_s2):,} / {len(true_s2_ids):,}"
)

log.info(
    f"True S3 records retrieved: "
    f"{len(matched_s3):,} / {len(true_s3_ids):,}"
)


# Sanity checks

missing_s2 = true_s2_ids - set(matched_s2["entity_id"])
missing_s3 = true_s3_ids - set(matched_s3["entity_id"])

if missing_s2:
    log.warning(
        f"Could not find {len(missing_s2):,} S2 IDs "
        f"in the S2 source file."
    )

if missing_s3:
    log.warning(
        f"Could not find {len(missing_s3):,} S3 IDs "
        f"in the S3 source file."
    )


# ============================================================
# 7. DETERMINE HOW MANY EXTRA RECORDS TO ADD
# ============================================================

section("SELECTING EXTRA S2/S3 RECORDS")

remaining_s2 = s2[
    ~s2["entity_id"].isin(true_s2_ids)
].copy()

remaining_s3 = s3[
    ~s3["entity_id"].isin(true_s3_ids)
].copy()

extra_s2_n = min(
    len(remaining_s2),
    max(len(matched_s2) * NEGATIVE_MULTIPLIER, 1000)
)

extra_s3_n = min(
    len(remaining_s3),
    max(len(matched_s3) * NEGATIVE_MULTIPLIER, 1000)
)

log.info(
    f"Available non-match S2 records: "
    f"{len(remaining_s2):,}"
)

log.info(
    f"Available non-match S3 records: "
    f"{len(remaining_s3):,}"
)

log.info(
    f"Selecting extra S2 records: {extra_s2_n:,}"
)

log.info(
    f"Selecting extra S3 records: {extra_s3_n:,}"
)


# ============================================================
# 8. SAMPLE EXTRA RECORDS
# ============================================================

section("SAMPLING EXTRA RECORDS")

extra_s2 = remaining_s2.sample(
    n=extra_s2_n,
    random_state=RANDOM_STATE
)

extra_s3 = remaining_s3.sample(
    n=extra_s3_n,
    random_state=RANDOM_STATE
)

log.info(f"Extra S2 sampled: {len(extra_s2):,}")
log.info(f"Extra S3 sampled: {len(extra_s3):,}")


# ============================================================
# 9. BUILD FINAL S2/S3 SAMPLE
# ============================================================

section("BUILDING FINAL SAMPLE")

sample_s2 = pd.concat(
    [matched_s2, extra_s2],
    ignore_index=True
)

sample_s3 = pd.concat(
    [matched_s3, extra_s3],
    ignore_index=True
)


# Shuffle so true matches aren't grouped together.

sample_s2 = sample_s2.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

sample_s3 = sample_s3.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)


log.info(
    f"Final S1 sample: {len(sample_s1):,}"
)

log.info(
    f"Final S2 sample: {len(sample_s2):,}"
)

log.info(
    f"Final S3 sample: {len(sample_s3):,}"
)


# ============================================================
# 10. CREATE OUTPUT DIRECTORY
# ============================================================

section("CREATING OUTPUT DIRECTORY")

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

log.info(f"Output directory: {OUTPUT_DIR}")


# ============================================================
# 11. SAVE FILES
# ============================================================

section("SAVING FILES")

s1_path = f"{OUTPUT_DIR}/sample_source1.tsv"
s2_path = f"{OUTPUT_DIR}/sample_source2.tsv"
s3_path = f"{OUTPUT_DIR}/sample_source3.tsv"
gt_path = f"{OUTPUT_DIR}/sample_ground_truth.tsv"


log.info(f"Writing {s1_path}")
sample_s1.to_csv(
    s1_path,
    sep="\t",
    index=False
)

log.info(f"Writing {s2_path}")
sample_s2.to_csv(
    s2_path,
    sep="\t",
    index=False
)

log.info(f"Writing {s3_path}")
sample_s3.to_csv(
    s3_path,
    sep="\t",
    index=False
)

log.info(f"Writing {gt_path}")
sample_gt.to_csv(
    gt_path,
    sep="\t",
    index=False
)


# ============================================================
# 12. FINAL SANITY CHECKS
# ============================================================

section("FINAL SANITY CHECKS")

# Every sampled S1 should have a ground-truth row.

assert len(sample_s1) == len(sample_gt), (
    "Number of sampled S1 records and ground-truth rows differ!"
)

# Every true S2 should be present.

assert true_s2_ids.issubset(
    set(sample_s2["entity_id"])
), "Some true S2 matches are missing from the sample!"

# Every true S3 should be present.

assert true_s3_ids.issubset(
    set(sample_s3["entity_id"])
), "Some true S3 matches are missing from the sample!"

# No duplicate IDs.

assert sample_s1["entity_id"].is_unique
assert sample_s2["entity_id"].is_unique
assert sample_s3["entity_id"].is_unique

log.info("✓ Every sampled S1 has ground truth")
log.info("✓ Every true S2 match is included")
log.info("✓ Every true S3 match is included")
log.info("✓ No duplicate S1 IDs")
log.info("✓ No duplicate S2 IDs")
log.info("✓ No duplicate S3 IDs")


# ============================================================
# 13. FINAL SUMMARY
# ============================================================

section("FINAL SUMMARY")

log.info(f"S1:              {len(sample_s1):,}")
log.info(f"S2:              {len(sample_s2):,}")
log.info(f"S3:              {len(sample_s3):,}")
log.info(f"S1 singletons:   {n_singletons:,}")
log.info(f"S1 with matches: {n_non_singletons:,}")
log.info(f"True S2:         {len(true_s2_ids):,}")
log.info(f"True S3:         {len(true_s3_ids):,}")

log.info("")
log.info("Output files:")
log.info(f"  {s1_path}")
log.info(f"  {s2_path}")
log.info(f"  {s3_path}")
log.info(f"  {gt_path}")

log.info("")
log.info("✓ SAMPLING COMPLETE")